# Documentation de la classe `WeatherData`

La classe `WeatherData` (`kadi.weather.data.WeatherData`) est responsable de l'acquisition, de la mise en cache (via SQLite global dans KadiPy) et de la normalisation des données météorologiques. Elle gère la transition transparente entre les requêtes en temps réel (API Open-Meteo) et les bases de données historiques locales (CHIRPS).

## 1. Initialisation
L'initialisation requiert uniquement une instance de la classe `Location`.

In [2]:
import os
from pathlib import Path
print(Path.cwd())
new_dir = Path("~/Bureau/kadipy/").expanduser().resolve()
os.chdir(new_dir)
print(Path.cwd())

from kadi.weather.location import Location
from kadi.weather.data import WeatherData


loc = Location(latitude=9.3333, longitude=2.6333, name='Parakou')
weather = WeatherData(loc)

print(f"Source des données à l'initialisation : {weather.data_source}")

/home/dels/Bureau/kadipy/docs/weather
/home/dels/Bureau/kadipy
Source des données à l'initialisation : none


## 2. Récupération des Prévisions (`fetch_forecast`)
**Méthode :** `fetch_forecast(days: int = 7, force_refresh: bool = False) -> pd.DataFrame`

Cette méthode tente de charger d'abord le cache local. Si le cache a expiré ou si `force_refresh=True` est utilisé, elle appellera l'API d'Open-Meteo pour récupérer des prévisions allant jusqu'à `days` jours.
Elle retourne un DataFrame `pandas` indexé par `date`.

In [ ]:
# Récupération des prévisions sur 3 jours
forecast_df = weather.fetch_forecast(days=3)

print(forecast_df.shape)

print(f"Source : {weather.data_source}")
display(forecast_df.head())

(3, 10)
Source : cached


,temperature_min,temperature_max,temperature_avg,precipitation,humidity,data_type,data_source,confidence,fetched_at,temperature_mean
date,,,,,,,,,,
2026-07-02,22.1,31.3,26.7,13.8,0.0,forecast,mock_api,1.0,2026-07-02T10:40:47.121083,26.7
2026-07-03,22.1,30.1,26.1,8.8,0.0,forecast,mock_api,1.0,2026-07-02T10:40:47.121083,26.1
2026-07-04,22.7,30.7,26.7,0.1,0.0,forecast,mock_api,1.0,2026-07-02T10:40:47.121083,26.7


## 3. Récupération de l'Historique (`fetch_historical`)
**Méthode :** `fetch_historical(months_back: int = 120, force_refresh: bool = False) -> pd.DataFrame`

Récupère l'historique météo. Le comportement technique de cette méthode est une aggrégation :
- Les données de **température** sont extraites d'Open-Meteo Historical.
- Les données de **précipitations** font un *fallback* sur les données locales du fichier raster **CHIRPS** (`~/.kadipy_cache/chirps_benin_daily.csv`) afin d'offrir une grande précision spatiale pour la pluviométrie.

Ces données combinées sont ensuite sauvegardées dans le cache SQLite `weather_data` de KadiPy.

In [28]:
# Historique sur 2 mois (approx. 60 jours)
historical_df = weather.fetch_historical(months_back=2)

print(historical_df.shape)

print(f"Source de l'historique : {weather.data_source}")
display(historical_df.head())

(61, 10)
Source de l'historique : cached


,temperature_min,temperature_max,temperature_avg,precipitation,humidity,data_type,data_source,confidence,fetched_at,temperature_mean
date,,,,,,,,,,
2026-05-01,23.5,36.2,29.85,0.0,0.0,historical,mock_api,1.0,2026-06-29T20:00:30.871927,29.85
2026-05-02,24.4,33.9,29.15,0.0,0.0,historical,mock_api,1.0,2026-06-29T20:00:30.871927,29.15
2026-05-03,24.9,34.8,29.85,0.0,0.0,historical,mock_api,1.0,2026-06-29T20:00:30.871927,29.85
2026-05-04,22.8,34.2,28.50,0.6,0.0,historical,mock_api,1.0,2026-06-29T20:00:30.871927,28.50
2026-05-05,23.2,35.3,29.25,20.1,0.0,historical,mock_api,1.0,2026-06-29T20:00:30.871927,29.25


## 4. Architecture Interne (Méthodes Privées)
En cas de développement sur ce module, voici les méthodes privées gérant la plomberie interne :
- `_fetch_forecast_data` et `_fetch_historical_data` : Réalisent les appels réseaux avec un mécanisme de *retry* configuré (tentatives et backoff) via `kadi._utils.network`.
- `_get_from_cache(start_date, end_date)` : Lit les données SQLite.
- `_save_to_cache(data, data_type)` : Exécute des `UPSERT` massifs dans SQLite pour rafraichir le cache.
- `_normalize_data` : S'assure que les dates sont bien formatées et utilisées comme index DataFrame.